In [ ]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
import sys
###
lib_path = [
    r'C:\Users\ikahbasi\OneDrive\Applications\GitHub\SeisRoutine',
    r'C:\Users\ikahb\OneDrive\Applications\GitHub\SeisRoutine',
]
for path in lib_path:
    sys.path.append(path)

In [ ]:
import SeisRoutine.seisbench as srsb
import SeisRoutine.config as srconf

In [ ]:
import seisbench.data as sbd
import seisbench.generate as sbg
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pprint

In [ ]:
def cmap(phase_hint):
    c = {
        'P': 'r',
        'S': 'b',
        'Pg': 'r',
        'Sg': 'b',
        'AML': 'c'
    }
    return c.get(phase_hint, 'y')

In [ ]:
def plot_3ch_with_phase(array_3c, metadata):
    fig, axes = plt.subplots(
        nrows=3, ncols=1,
        figsize=(10, 5),
        sharex=True,
        gridspec_kw={'hspace': 0, 'wspace': 0},
        subplot_kw={'adjustable': 'box'},
    )
    for ax, array_1c in zip(axes, array_3c):
        ax.plot(array_1c, c='k', lw=0.5)
        ax.patch.set_visible(False)
        ax.axis('off')
    key2hint_map = srsb.dataset.build_phase_mapper(list(metadata.keys()))
    for key, phase_hint in key2hint_map.items():
        for ax in axes:
            ax.axvline(
                x=metadata[key], lw=2, c=cmap(phase_hint), label=phase_hint
            )
    axes[0].legend()


In [ ]:
def plot_generator(
        n=None,
        generator=None,
        dataset=None,
        target_keys=None,
    ):
    if n is None:
        n = np.random.randint(len(generator))
    print(f'{n=}')
    # with pd.option_context('display.max_rows', None):
    # print(data.metadata.iloc[n])
    sample = generator[n]
    print(dataset.metadata.iloc[n][target_keys])
    fig, axs = plt.subplots(
        nrows=2, ncols=1,
        sharex=True,
        figsize=(15, 5),
        gridspec_kw={
            "hspace": 0,
            "height_ratios": [3, 1],
        }
    )
    axs[0].plot(sample["X"].T)
    axs[1].plot(sample["y"].T, label=['N', 'P', 'S'])
    plt.legend()

### Loading the dataset

Now that the dataset conversion is finished, we can check it by simply loading it. Here we load the dataset, print the metadata and visualize the first waveform together with the annotated pick.

In [ ]:
timestamp = srconf.timestamp()

cfg_projects = srconf.Config.load('./Configs/Projects.yml')
cfg_project = cfg_projects.extra_parameters

cfg = srconf.Config.load(
    file_path=cfg_project.parameters_config_path,
    resolve=True,
)
context={
    "timestamp": timestamp,
    "project": cfg_project,
}
cfg.resolve(context=context)

In [ ]:
# path = r"D:\DataSets-Local\1405-04-03\Merged_Dataset_2026-06-24T15-15-22"

cfg.dataset.path = Path(cfg.dataset.path)
dataset = sbd.WaveformDataset(
    path=cfg.dataset.path,
    **cfg.dataset.data_format.to_dict()
)
dataset.metadata['split'] = 'train'

In [ ]:
print(list(dataset.metadata.keys()))

In [ ]:
list(filter(lambda x: x.startswith('trace_name'), dataset.metadata.keys()))

In [ ]:
print("Training examples:", len(dataset.train()))
print("Development examples:", len(dataset.dev()))
print("Test examples:", len(dataset.test()))

In [ ]:
pprint.pp(dataset.metadata['station_code'].value_counts().to_dict())

In [ ]:
dataset.metadata['station_network_code'].value_counts()

In [ ]:
keys = dataset.metadata.keys()
dataset.metadata[[key for key in keys if key.startswith('station')]]

In [ ]:
# range_ii = [0, 4]
# for ii, metadata in dataset.metadata.iterrows():
#     if range_ii[0] < ii <= range_ii[1]:
#         # print(metadata)
#         fig = plt.figure(figsize=(7, 2.5))
#         ax = fig.add_subplot(111)
#         trace = dataset.get_waveforms(ii)
#         print(trace.shape)
#         ax.plot(trace.T, lw=0.3)
#         targets = [key for key in metadata.keys() if 'arrival' in key]
#         targets = [key for key in targets if ~np.isnan(metadata[key])]
#         # print(targets, metadata[targets])
#         for target in targets:
#             phase_hint = target.split('_')[1]
#             ax.axvline(metadata[target], lw=3, c=cmap(phase_hint), label=phase_hint)
#         plt.legend()
#         plt.show()

In [ ]:
range_ii = [0, 4]
for ii, metadata in dataset.metadata.iterrows():
    if range_ii[0] < ii <= range_ii[1]:
        array_3c = dataset.get_waveforms(ii)
        plot_3ch_with_phase(array_3c, metadata)

In [ ]:
phase_dict = srsb.dataset.build_phase_mapper(dataset.metadata)
phase_dict

In [ ]:
generator = sbg.GenericGenerator(dataset)

augmentations = [
    srsb.dataset.Tapering(),
    sbg.GaussianNoise(scale=(0.01, 0.02), key='X'),
    sbg.RandomWindow(windowlen=3001, strategy="pad"),
    sbg.Normalize(demean_axis=-1, amp_norm_axis=-1, amp_norm_type="peak"),
    sbg.ChangeDtype(np.float32),
    sbg.ProbabilisticLabeller(label_columns=phase_dict, sigma=30, dim=0)
]

generator.add_augmentations(augmentations)

In [ ]:
plot_generator(n=3347, generator=generator, dataset=dataset, target_keys=targets)

In [ ]:
plot_generator(n=1, generator=generator, dataset=dataset, target_keys=targets)

In [ ]:
plot_generator(n=15229, generator=generator, dataset=dataset, target_keys=targets)

In [ ]:
plot_generator(n=15351, generator=generator, dataset=dataset, target_keys=targets)

In [ ]:
array_3c[0]

In [ ]:
targets = srsb.dataset.build_phase_mapper(list(metadata.keys()))
targets

In [ ]:
ii = 4
array_3c = dataset.get_waveforms(ii)
plot_3ch_with_phase(array_3c, metadata)